# Числовые признаки пар

Здесь исследуются характеристики и рассчитываются признаки массы и чисел из названий. Общая таблица собирается только в `Feature_assembly.ipynb`.


## 1. Импорты


In [97]:
import numpy as np
import pandas as pd
from sklearn import metrics
import json
import re


## 2. Загрузка пар


In [12]:
pairs = pd.read_parquet('data/pairs.parquet')
pairs.head()

,id1,id2,target,category,name_1,name_2,attributes_1,attributes_2
0,32,188978721454,0.0,Аптека,очки с футляром-змейка fedrov мод. 1047 цвет 3 с линзами romeo 1.56 hmc/emi -2.00 рц 62-64,"фотохромные очки с футляром-змейка fedrov мод. 1047 цвет 3 с линзами romeo 1.56 fast photocolor brown, hmc+ -3.00 рц 64-66","{""диоптрии"":""- 2.00"",""цвет оправы"":""золотистый, коричневый"",""назначение"":""корректирующие"",""конструкция"":""ободковая"",""материал оправы"":""комбинированный"",""материал линз"":""пластик"",""форма оправы"":""бабочка"",""покрытие линз"":""упрочняющее, водоотталкивающее и антистатическое, металлизированное (защита от электромагнитных волн) uv-380, просветляющее"",""пол"":""женские"",""ширина линзы"":""56 мм"",""высота линзы"":""52 мм"",""межцентровое расстояние"":""62-64"",""длина заушника"":""144 мм""}","{""наличие флексов"":""да"",""затемнение"":""коричневый фотохром 80%"",""бренд"":""fedrov"",""цвет товара"":""золотой"",""название цвета"":""brown & gold"",""страна-изготовитель"":""россия"",""размер заушника, мм"":""144"",""коллекция (год)"":""2022"",""конструкция очков"":""ободковые"",""форма оправы"":""бабочки"",""материал оправы очков"":""комбинированный"",""материал линз"":""пластик"",""стиль оправы"":""повседневный"",""форма лица"":""квадратная"",""покрытие линз"":""упрочняющее, просветляющее, металлизированное (защита от электромагнитных волн) emi, uv400, водоотталкивающее и антистатическое"",""высота линзы, мм"":""52"",""ширина линзы, мм"":""56"",""размер моста на переносице, мм"":""15"",""расстояние между оптическими центрами"":""64-66"",""оптическая сила"":""-3.00"",""комплектация"":""очки fedrov с полимерными линзами romeo photocolor brown 1.56 hmc+, микрофибровая салфетка для ухода, футляр-змейка."",""пол ребенка"":""унисекс"",""пол"":""женский"",""общая ширина, мм"":""139"",""тип"":""очки для дали""}"
1,34,197568543106,0.0,Красота и гигиена,"зубная щетка r.o.c.s. black edition classic, черная/красная","зубная щетка r.o.c.s. red edition classic, средняя","{""бренд"":""r.o.c.s."",""тип"":""зубная щетка"",""уровень жесткости"":""средняя"",""особенности"":""нескользящая ручка"",""минимальный возраст"":""от 16 лет"",""материал щетины"":""синтетическая"",""материал корпуса"":""пластик"",""длина головки"":""18.4 мм"",""способ применения"":""скошенная подстрижка щетины позволяет зубной щётке проникать в труднодоступные места и к дальним зубам. благодаря технологии тройной полировки кончиков щетинки идеально закруглённые и гладкие, поэтому безопасны для дёсен.тонкая лёгкая ручка из безопасного высококачественного пластика предотвращает излишнее давление при чистике. кроме того, её гладкая поверхность без дополнительных деталей препятствует размножению бактерий, что обеспечивает гигиеничность щётки."",""дополнительная информация"":""скошенная подстрижка щетины позволяет зубной щётке проникать в труднодоступные места и к дальним зубам. благодаря технологии тройной полировки кончиков щетинки идеально закруглённые и гладкие, поэтому безопасны для дёсен.тонкая лёгкая ручка из безопасного высококачественного пластика предотвращает излишнее давление при чистике. кроме того, её гладкая поверхность без дополнительных деталей препятствует размножению бактерий, что обеспечивает гигиеничность щётки"",""цвет товара"":""черная/красная"",""количество упаковок"":""другое""}","{""целевая аудитория"":""взрослая"",""бренд"":""r.o.c.s."",""цвет товара"":""красный"",""страна-изготовитель"":""россия"",""жесткость щетины"":""средняя"",""тип"":""зубная щетка""}"
2,37,661425040668,0.0,Обувь,сандалии fre gamo,fre gamo / сандалии,"{""бренд в одежде и обуви"":""fre gamo"",""страна бренда"":""россия"",""коллекция"":""весна-лето 2021"",""цвет товара"":""черный"",""название цвета"":""черный"",""страна-изготовитель"":""китай"",""материал верха"":""искусственная кожа"",""материал"":""искусственная кожа"",""российский размер (обуви)"":""43"",""сезон"":""лето"",""пол"":""мужской"",""размер производителя"":""43"",""материал подошвы"":""резин

## 3. Исследование ключей характеристик


In [ ]:
def find_measurement_keys(attributes):
    attributes = json.loads(attributes)

    found_keys = []
    weight = ['вес', 'масса', 'объем', 'объём']

    for key in attributes:
        if any(word in key.lower() for word in weight):
            found_keys.append(key)
    return found_keys


In [55]:
measurement_key_counts_1 = pairs['attributes_1'].apply(find_measurement_keys).explode().value_counts()
measurement_key_counts_2 = pairs['attributes_2'].apply(find_measurement_keys).explode().value_counts()

In [61]:
key_counts_table = pd.concat(
    [measurement_key_counts_1, measurement_key_counts_2],
    axis=1
)

key_counts_table.columns = ['attributes_1', 'attributes_2']
key_counts_table = key_counts_table.fillna(0).astype(int)
key_counts_table.index.name = 'поле'

key_counts_table.head(100)

,attributes_1,attributes_2
поле,,
"вес товара, г",53581,53159
вес товара с упаковкой (г),14988,15069
"объем, мл",13464,13235
вес,11829,11676
вес товара без упаковки (г),10840,10867
вес с упаковкой (кг),9566,9231
"средний вес изделия, гр",9285,9383
вес в транспортной упаковке,8346,8099
"вес, кг",7586,7589


## 4. Извлечение массы в граммах

Используется первое корректное поле в порядке ключей исходных характеристик. Алгоритм извлечения сохранён без изменений.


In [ ]:
def get_product_weight(attributes):
    attributes = json.loads(attributes)

    weight_fields = {
        'вес товара, г': 1,
        'вес товара без упаковки (г)': 1,
        'вес, г': 1,
        'вес, в граммах': 1,
        'вес, гр': 1,
        'вес, в гр': 1,
        'вес (г)': 1,
        'вес без упаковки (кг)': 1000,
        'вес, кг': 1000,
        'вес, в килограммах': 1000,
        'вес, в кг': 1000,
        'вес (кг)': 1000,
        'вес товара (кг)': 1000,
        'вес нетто (кг)': 1000
    }

    for key in attributes:
        if key in weight_fields:
            weight = attributes[key]
            weight = pd.to_numeric(weight, errors='coerce')
            weight = weight * weight_fields[key]
            if pd.notna(weight) and weight > 0:
                return weight
    return None


In [63]:
weights_g_1 = pairs['attributes_1'].apply(get_product_weight)
weights_g_2 = pairs['attributes_2'].apply(get_product_weight)


In [71]:
mask = weights_g_1.notna() & weights_g_2.notna()


weight_check = pairs[['name_1', 'name_2', 'attributes_1', 'attributes_2', 'target']].copy()
weight_check['weight_g_1'] = weights_g_1
weight_check['weight_g_2'] = weights_g_2

weight_check.loc[mask].head(10)

,name_1,name_2,attributes_1,attributes_2,target,weight_g_1,weight_g_2
23,видеодомофон tantos elly,видеодомофон tantos elly - белый,"{""угол обзора по вертикали, °"":""70"",""бренд"":""tantos"",""разрешение камеры"":""800 твл"",""цвет товара"":""белый"",""конструктивные особенности"":""сенсорный экран"",""тип управления"":""электронное"",""страна-изготовитель"":""китай"",""декоративное покрытие корпуса"":""пластик"",""размеры, мм"":""118 х 180 х 27"",""особенности"":""интерком"",""основной материал корпуса видеодомофона"":""пластик"",""слот для карты памяти"":""нет"",""число подключаемых камер"":""2"",""число подключаемых мониторов"":""4"",""число подключаемых вызывных панелей"":""2"",""количество мелодий"":""16"",""камера"":""1.3 мпикс"",""комплектация"":""монитор, кронштейн, 2 дюбеля, 2 шурупа, разъем 5 контактов, 2 разъем 4 контакта, разъем 4 контакта, инструкция"",""питание от"":""сети 220"",""мощность, вт"":""70"",""диагональ экрана, дюймы"":""4.5"",""разрешение экрана"":""480x272"",""тип"":""видеодомофон"",""вид домофона"":""без трубки"",""цветность дисплея"":""цветной"",""гарантийный срок"":""36 месяцев"",""вес товара, г"":""288"",""подключение"":""проводное""}","{""модель"":""tantos elly"",""бренд"":""tantos"",""артикул производителя"":""dp-1006-ts"",""тип видеосигнала"":""аналоговый"",""диагональ экрана, в дюймах"":""4.3"",""разрешение монитора, пикселей"":""480x270"",""число мониторов в системе"":""4"",""число видеокамер в системе"":""2"",""число вызывных панелей в системе"":""2"",""кнопка вызова панели"":""сенсорная"",""угол обзора"":""0"",""страна-производитель"":""китай"",""гарантия производителя, в месяцах"":""36"",""количество каналов детектора движения"":""0"",""напряжение питания"":""220"",""минимальная рабочая температура"":""0"",""максимальная рабочая температура"":""40"",""функции и особенности"":""интерком"",""комплект поставки"":""монитор"",""длина, в сантиметрах"":""12"",""ширина, в сантиметрах"":""18"",""высота, в сантиметрах"":""3"",""вес, в килограммах"":""0.29"",""код товара"":""600002046903""}",1.0,288.0,290.0
94,маска глиняная для лица с экстрактом розы и гиалуроновой кислотой mary & may rose hyaluronic hydra glow wash off pack 125 гр,глиняная маска mary&may с экстрактом розы и гиалуроновой кислотой rose 125 г,"{""возрастной диапазон"":""от 25 лет"",""способ применения"":""нанести маску на кожу, избегая области глаз и губ, оставить на 15 минут, смыть мягкими массирующими движениями"",""область использования"":""лицо"",""бренд"":""mary&may"",""состав"":""water, diglycerin, glycerin, kaolin, bentonite, propanediol, zea mays (corn) starch, 1,2-hexanediol, rosa damascena flower water, panthenol, rosa gallica flower powder, xanthan gum, sodium acrylate/sodium acryloyldimethyl taurate copolymer, polyisobutene, betaine, butylene glycol, fragrance (parfum), ethylhexylglycerin, dipropylene glycol, ci77491, disodium edta, caprylyl/capryl glucoside, sorbitan oleate, allantoin, hydrogenated lecithin, centella asiatica extract, ficus carica (fig) fruit extract, ceramide np, sodium hyaluronate, tocopherol, hyaluronic acid, sodium hyaluronate crosspolymer, hydroxypropyltrimonium hyaluronate, sodium acetylated hyaluronate."",""вид маски"":""минеральная"",""страна-изготовитель"":""южная корея"",""особенности состава"":""не тестируется на животных"",""бьюти-ингредиент"":""гиалуроновая кислота"",""упаковка"":""банка"",""единиц в одном товаре"":""1"",""применение"":""дневной уход"",""эффект от использования уходовой косметики"":""восстановление"",""тип кожи"":""для всех типов кожи"",""тип"":""маска косметическая"",""объем, мл"":""125"",""вес, г"":""125""}","{""бренд"":""mary&may"",""артикул производителя"":""8809670681586"",""тип"":""глиняная маска"",""тип кожи"":""для комбинированной кожи; для нормальной кожи"",""эффект"":""питательный; смягчающий; улучшение цвета лица"",""объем, мл"":""125"",""вес, г"":""200"",""количество в упаковке"":""1"",""состав"":""water, diglycerin, glycerin, kaolin, bentonite, propanediol, ze

## 5. Относительная разница массы


In [73]:
weight_rel_diff = (
    (weights_g_1 - weights_g_2).abs()
    / pd.concat([weights_g_1, weights_g_2], axis=1).max(axis=1)
)

In [76]:
pairs['weight_rel_diff'] = weight_rel_diff

### Сохранение признака массы


In [77]:
pairs[['id1', 'id2', 'target', 'weight_rel_diff']].to_parquet(
    'data/weight_features.parquet',
    index=False
)

## 6. Извлечение чисел из названий


In [ ]:
def extract_numbers(name):
    numbers = set()

    founds = re.findall(
        r'(?<![0-9a-zа-яё.,])[+-]\s*\d+(?:[.,]\d+)?|\d+(?:[.,]\d+)?',
        name.lower()
    )

    for found in founds:
        found = found.replace(',', '.').replace(' ', '')
        found = float(found)
        numbers.add(found)
    return numbers


In [81]:
pairs['numbers_1'] = pairs['name_1'].apply(extract_numbers)
pairs['numbers_2'] = pairs['name_2'].apply(extract_numbers)



In [84]:
pairs[['name_1', 'name_2', 'numbers_1', 'numbers_2', 'target']].head(10)

,name_1,name_2,numbers_1,numbers_2,target
0,очки с футляром-змейка fedrov мод. 1047 цвет 3 с линзами romeo 1.56 hmc/emi -2.00 рц 62-64,"фотохромные очки с футляром-змейка fedrov мод. 1047 цвет 3 с линзами romeo 1.56 fast photocolor brown, hmc+ -3.00 рц 64-66","{64.0, 1.56, 3.0, -2.0, 1047.0, 62.0}","{64.0, 1.56, 66.0, 3.0, 1047.0, -3.0}",0.0
1,"зубная щетка r.o.c.s. black edition classic, черная/красная","зубная щетка r.o.c.s. red edition classic, средняя",{},{},0.0
2,сандалии fre gamo,fre gamo / сандалии,{},{},0.0
3,"acuvue астигматические линзы oasys for astigmatism,ось:10_-0.75, 6 шт., -0.75 / 8.6/-0.75/10, 2 недели",acuvue / контактные линзы acuvue oasys for astigmatism (6 линз) r 8.6 d +4.75 cyl -0.75 ax 10,"{-0.75, 2.0, 6.0, 8.6, 10.0}","{-0.75, 4.75, 6.0, 8.6, 10.0}",0.0
4,"дворцовый пуэр (5 лет), 500 г",чайная линия ореховый пуэр (500 гр) вес: 500 гр,"{500.0, 5.0}",{500.0},0.0
5,eclat парфюм туалетная вода 100 мл духи 100 мл,духи lux / духи эклат красота ланвин и туалетная вода,{100.0},{},0.0
6,victor reinz прокладка впускного коллектора арт. 703315700,victor reinz / втулка шпильки клапанной крышки victor reinz 70-28778-00,{703315700.0},"{0.0, 28778.0, 70.0}",0.0
7,"stellox диск тормозной, арт. 8500892sx","stellox / диск торм. задний fiat ducato,peugeot boxer,citroen jumper",{8500892.0},{},0.0
8,микроволновая печь соло lg ms-20r42d,"lg / микроволновая печь ms20r42d, 20 литров","{42.0, 20.0}","{42.0, 20.0}",1.0
9,"сухой корм для взрослых собак крупных пород мираторг black angus с говядиной 2,6 кг.","мираторг / extra meat dog large говядина 2,6 кг",{2.6},{2.6},1.0


## 7. Жаккар по числам


In [ ]:
def numbers_jaccard(numbers_1, numbers_2):
    union = numbers_1 | numbers_2
    intersection = numbers_1 & numbers_2
    if not union:
        return None
    else:
        return len(intersection) / len(union)


In [92]:
pairs['numbers_jaccard'] = pairs.apply(
    lambda row: numbers_jaccard(row['numbers_1'], row['numbers_2']),
    axis=1
)




## 8. Оценка числового Жаккара

ROC-AUC считается только для пар с заполненным признаком.


In [99]:
mask_numbers = pairs['numbers_jaccard'].notna()

pairs.loc[mask_numbers].head(10)
roc_auc = metrics.roc_auc_score(pairs.loc[mask_numbers, 'target'],pairs.loc[mask_numbers, 'numbers_jaccard'] )
print(roc_auc)

0.6285314531779773


## 9. Сохранение числового Жаккара


In [ ]:
pairs[['id1', 'id2', 'target', 'numbers_jaccard']].to_parquet(
    'data/number_features.parquet', index=False
)


## Выводы

- Масса доступна у обоих товаров для 6660 пар (около 1,8%).
- `weight_rel_diff`: 0 означает одинаковую массу; большее значение — большее относительное различие. При недостатке данных остаётся пропуск.
- `numbers_jaccard` заполнен для 304 218 пар. ROC-AUC — 0,6285 на этих парах.
- Числа из названий сравниваются без нормализации единиц. При отсутствии чисел с обеих сторон остаётся пропуск.
- Результаты: `data/weight_features.parquet` и `data/number_features.parquet`.
